# Phase 3: Exploratory Data Analysis (EDA) - Retina Module

This notebook loads the cached dataset statistics, generates descriptive figures, and establishes baseline configurations for model training.
The analysis is strictly **read-only** and performs no image modifications.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Imports from src.config
from src.config import (
    EDA_STATISTICS_CSV,
    EDA_SUMMARY_JSON,
    PREPROCESSING_RECOMMENDATIONS_JSON,
    CLASS_NAMES,
    TRAIN_IMAGES,
    SEED
)

np.random.seed(SEED)
print("Libraries loaded and random seed initialized.")

## 1. Dataset Overview
We load the global summary JSON containing versions, environment configurations, and the deterministic dataset fingerprint.

In [ ]:
with open(EDA_SUMMARY_JSON, 'r') as f:
    summary = json.load(f)

print(f"Dataset: {summary['dataset']}")
print(f"EDA Version: {summary['eda_version']}")
print(f"Dataset Fingerprint (SHA-256): {summary['dataset_fingerprint']}")
print(f"Total Labeled Training Images: {summary['total_training_images']}")
print(f"Missing Images: {summary['missing_images']} | Corrupted: {summary['corrupted_images']} | Invalid Channels: {summary['invalid_channel_images']}")

## 2. Class Distribution Analysis
Check target label proportions to evaluate class imbalances.

In [ ]:
df = pd.read_csv(EDA_STATISTICS_CSV)
counts = df['diagnosis'].value_counts().sort_index()
for cid, (name, count) in enumerate(zip(CLASS_NAMES, counts)):
    pct = (count / len(df)) * 100
    print(f"Class {cid} ({name}): {count} images ({pct:.2f}%)")

## 3. Resolution, Aspect Ratio, & Size Statistics
Identify the image shapes, crop factors, and disk footprints.

In [ ]:
print(f"Width  -> Min: {df['width'].min()} | Max: {df['width'].max()} | Avg: {df['width'].mean():.1f} pixels")
print(f"Height -> Min: {df['height'].min()} | Max: {df['height'].max()} | Avg: {df['height'].mean():.1f} pixels")
print(f"Aspect Ratio -> Avg: {df['aspect_ratio'].mean():.2f} | Std: {df['aspect_ratio'].std():.2f}")
print(f"File Size    -> Avg: {df['filesize_kb'].mean():.1f} KB | Std: {df['filesize_kb'].std():.1f} KB")

## 4. Grayscale Brightness & Laplacian Sharpness Analysis
Verify the brightness and focus distributions across the dataset.

In [ ]:
print(f"Average Grayscale Brightness: {df['brightness_score'].mean():.2f} (std: {df['brightness_score'].std():.2f})")
print(f"Average Laplacian Sharpness Score: {df['blur_score'].mean():.2f} (std: {df['blur_score'].std():.2f})")
print(f"Darkest 5% Threshold: < {summary['thresholds']['brightness_5pct']:.2f}")
print(f"Blurriest 5% Threshold: < {summary['thresholds']['blur_5pct']:.2f}")

## 5. Visual Duplicate Verification
Verifies if there are duplicate image pairs based on pixel contents hashes.

In [ ]:
dup_path = Path(EDA_STATISTICS_CSV).parent / 'duplicate_images.csv'
if dup_path.exists():
    dup_df = pd.read_csv(dup_path)
    print(f"Total exact visual duplicate image pairs detected: {len(dup_df)}")
else:
    print("No visual duplicates verification file found.")

## 6. Continuous Image Quality Score (Q)
Loads the continuous Image Quality score computed from normalized resolution, brightness, and sharpness.

In [ ]:
print(f"Average image quality score Q: {df['quality_score'].mean():.4f} (std: {df['quality_score'].std():.4f})")
print(f"Minimum quality score: {df['quality_score'].min():.4f} | Maximum: {df['quality_score'].max():.4f}")

## 7. RGB Global Normalization Statistics
Compute dataset-specific channel-wise mean and standard deviations.

In [ ]:
print(f"Global RGB Mean: R={summary['rgb_mean'][0]:.6f}, G={summary['rgb_mean'][1]:.6f}, B={summary['rgb_mean'][2]:.6f}")
print(f"Global RGB Std:  R={summary['rgb_std'][0]:.6f}, G={summary['rgb_std'][1]:.6f}, B={summary['rgb_std'][2]:.6f}")

## 8. Preprocessing Recommendations
Review the final configuration guidelines derived from the EDA pipeline.

In [ ]:
with open(PREPROCESSING_RECOMMENDATIONS_JSON, 'r') as f:
    recs = json.load(f)
print(json.dumps(recs, indent=2))